# Build Data Pipelines with Lakeflow Spark Declarative Pipelines

Training notebook covering Lakeflow Spark Declarative Pipelines (SDP) with the `dp` Python API: streaming tables, materialized views, expectations, Auto CDC, Auto Loader integration, medallion architecture, and pipeline operations — with side-by-side comparisons to regular Spark Structured Streaming.

## 📋 What We'll Cover

1. **What is Lakeflow Spark Declarative Pipelines** — purpose, how it differs from regular Spark jobs
2. **Key Concepts** — streaming tables, materialized views, temporary views, expectations
3. **Python API** — `from pyspark import pipelines as dp`
4. **Example 1: Regular Streaming vs SDP** — bronze/silver ingestion both ways
5. **Example 2: Streaming MERGE vs Auto CDC** — upserts and deletes both ways
6. **Medallion Architecture in SDP** — bronze → silver → gold pattern
7. **Data Quality with Expectations** — `@dp.expect`, `expect_or_drop`, `expect_or_fail`
8. **This Training's Pipeline** — config-driven pipeline file and what it does
9. **Decision Matrix** — when to choose SDP vs regular streaming

## 1. What is Lakeflow Spark Declarative Pipelines (SDP)

Lakeflow Spark Declarative Pipelines is Databricks' **declarative ETL framework**. Instead of writing imperative Spark code that manually manages reads, writes, checkpoints, and retries, you **declare what tables you want** and SDP figures out how to build and keep them up to date.

### SDP vs Regular Spark Job

| Aspect | Regular Spark Job | Lakeflow SDP |
|---|---|---|
| **Write mode** | You manage `overwrite` / `append` / `merge` | Declared automatically from the function type |
| **Checkpoints** | You create and manage checkpoint paths | Managed by SDP automatically |
| **Dependencies** | You handle ordering manually | SDP builds the DAG from `spark.read.table()` / `spark.readStream.table()` references |
| **Data quality** | Custom `filter` / `assert` code | Built-in expectations with metrics tracking |
| **Retries on failure** | Job retry policy only | SDP retries at the table level |
| **Lineage** | Not automatic | Auto-tracked in Unity Catalog |

### How SDP works

```
You write Python functions ──────────────► SDP builds an execution DAG
                                            │
   @dp.table                                ▼
   def bronze():                    Streaming Table (incremental)
       return spark.readStream...   Materialized View (full recompute)
                                            │
   @dp.materialized_view                    ▼
   def gold():                      Delta table in Unity Catalog
       return spark.read.table("silver")...
```

> 📛 **Table names:** each decorated function becomes one dataset. Its name is the **function name** (`bronze`, `gold` above) unless you pass `name="..."` to the decorator. See **Section 2 → How datasets are named**.

## 2. Key Concepts

### Streaming Table
A table populated **incrementally** — each pipeline run processes only new data. Backed by Structured Streaming. Use for:
* Raw ingestion from files, Kafka, CDC
* Any source where you only want to process new records

```python
from pyspark import pipelines as dp

@dp.table                      # name -> my_streaming_table (function name)
def my_streaming_table():
    return spark.readStream.format("cloudFiles").load("/path/to/files")
```

### Materialized View
A table that is **fully recomputed** on each run. Use for:
* Aggregations, joins, and lookups where the full result set must be consistent
* Gold-layer reporting tables

```python
@dp.materialized_view          # name -> my_materialized_view (function name)
def my_materialized_view():
    return spark.read.table("my_streaming_table").groupBy("key").count()
```

### Temporary View
A virtual table visible only within the same pipeline run. Not persisted to Unity Catalog. Use for intermediate transformations you don't need to expose.

```python
@dp.temporary_view             # name -> my_temp_view (pipeline-only, not in Unity Catalog)
def my_temp_view():
    return spark.read.table("my_streaming_table").filter(col("status") == "active")
```

### How datasets are named

| How you write it | Dataset name | Published to Unity Catalog as |
|---|---|---|
| `@dp.table` on `def customer_bronze():` | `customer_bronze` (function name) | `<pipeline catalog>.<pipeline schema>.customer_bronze` |
| `@dp.table(name="cust_raw")` on any function | `cust_raw` (`name=` wins over the function name) | `<pipeline catalog>.<pipeline schema>.cust_raw` |
| `@dp.table(name="training.silver.customer_silver")` | fully qualified name | `training.silver.customer_silver` (overrides the pipeline default schema) |
| `@dp.materialized_view(name="customer_gold")` | `customer_gold` | `<pipeline catalog>.<pipeline schema>.customer_gold` |
| `@dp.temporary_view` on `def my_temp_view():` | `my_temp_view` | **Not published** — only visible inside the pipeline |
| `dp.create_streaming_table("customer_current")` | `customer_current` (first argument) | `<pipeline catalog>.<pipeline schema>.customer_current` |

* The **catalog and schema** come from the pipeline settings; code normally uses only the short name.
* Other datasets read it by that same name: `spark.read.table("customer_bronze")`.
* Each name can be defined **only once** per pipeline.
* Tip: set `name=` explicitly when the function name differs from the table you want, or when names come from config (Section 8).

### Key rule: reference pipeline tables with standard Spark readers
When reading a table defined **within the same pipeline**, use the table name directly (no `LIVE.` prefix):
* `spark.readStream.table("table_name")` for streaming reads
* `spark.read.table("table_name")` for batch reads

## 3. Python API

Pipelines are defined in Python with `from pyspark import pipelines as dp`, the API of the open-source Apache Spark Declarative Pipelines standard. Reads of other pipeline tables use standard `spark.read.table()` / `spark.readStream.table()`.

### Example
```python
from pyspark import pipelines as dp
from pyspark.sql.functions import col

@dp.table(comment="Bronze customer data")       # no name= -> table name is customer_bronze
def customer_bronze():
    return spark.readStream.format("cloudFiles").load("/path")

@dp.expect_or_drop("valid_key", "c_custkey IS NOT NULL")
@dp.table(comment="Cleaned customer data")      # no name= -> table name is customer_silver
def customer_silver():
    # reads the table above by its name
    return spark.readStream.table("customer_bronze").filter(col("c_acctbal") >= 0)
```

### Core decorators

| Decorator | Creates | Name set by |
|---|---|---|
| `@dp.table` + `readStream` | Streaming table | `name=` or function name |
| `@dp.materialized_view` + `read` | Materialized view | `name=` or function name |
| `@dp.temporary_view` | Temporary view | `name=` or function name |
| `@dp.expect` | Expectation (warn) | first argument = expectation name (not a table) |
| `@dp.expect_or_drop` | Expectation (drop row) | first argument = expectation name |
| `@dp.expect_or_fail` | Expectation (fail pipeline) | first argument = expectation name |

> 💡 This training uses the `dp` API throughout. The table above serves as the quick reference.

## 4. Example 1 — Regular Spark Streaming vs Declarative Pipeline

Same task both ways: ingest customer CSV files with Auto Loader into **bronze**, then build a cleaned **silver** table that drops rows without a key.

### A) Regular Spark Structured Streaming (imperative)

```python
from pyspark.sql.functions import col, current_timestamp

source_path = "abfss://inboundfiles@unityms.dfs.core.windows.net/customer"
chk = "/Volumes/training/bronze/checkpoints"

# ---- Bronze: you manage schema location, checkpoint, trigger and target ----
bronze_q = (
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", f"{chk}/customer_bronze/_schema")
        .load(source_path)
        .withColumn("_ingest_ts", current_timestamp())
    .writeStream
        .option("checkpointLocation", f"{chk}/customer_bronze")
        .trigger(availableNow=True)
        .toTable("training.bronze.customer_bronze")   # table name: full name hard-coded here
)
bronze_q.awaitTermination()          # you enforce the order: bronze before silver

# ---- Silver: data quality is hand-written filter logic, no metrics ----
silver_q = (
    spark.readStream.table("training.bronze.customer_bronze")
        .filter(col("c_custkey").isNotNull())
    .writeStream
        .option("checkpointLocation", f"{chk}/customer_silver")
        .trigger(availableNow=True)
        .toTable("training.bronze.customer_silver")   # table name: full name hard-coded here
)
silver_q.awaitTermination()
```

### B) Lakeflow Spark Declarative Pipeline (declarative)

```python
from pyspark import pipelines as dp
from pyspark.sql.functions import current_timestamp

source_path = "abfss://inboundfiles@unityms.dfs.core.windows.net/customer"

@dp.table(name="customer_bronze", comment="Raw customer files")   # table name: customer_bronze
def customer_bronze():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .load(source_path)
            .withColumn("_ingest_ts", current_timestamp())
    )

@dp.expect_or_drop("valid_key", "c_custkey IS NOT NULL")   # rule + metrics
@dp.table(name="customer_silver", comment="Cleaned customers")   # table name: customer_silver
def customer_silver():
    return spark.readStream.table("customer_bronze")         # reads by name -> dependency inferred
```

### What changed

| Concern | Regular streaming | Declarative pipeline |
|---|---|---|
| Checkpoint & schema location | You pick paths per table | Managed automatically |
| Write (`writeStream` / `toTable`) | You write it | Not written — the function just returns a DataFrame |
| Trigger | `.trigger(availableNow=True)` in code | Pipeline mode (Triggered / Continuous) in settings |
| Order bronze → silver | `awaitTermination()` or separate job tasks | DAG inferred from `spark.readStream.table()` |
| Table name | Full name in `toTable("training.bronze.customer_bronze")` | `name="customer_bronze"` (or function name) |
| Catalog / schema | Hard-coded in `toTable()` | Set once in pipeline settings |
| Data quality | `filter()` — dropped rows are invisible | `@dp.expect_or_drop` — drop counts in the UI and event log |
| Reprocess everything | Delete checkpoints + tables manually | **Full refresh** button |

> Note: pipeline code runs only inside a pipeline update. Running these `@dp` cells in a notebook just registers the functions.

## 5. Example 2 — Streaming MERGE (`foreachBatch`) vs Auto CDC

Task: a change feed `customer_changes` delivers inserts, updates and deletes (columns: `c_custkey`, customer attributes, `operation`, `event_ts`). Keep a `customer_current` table with the **latest row per customer** (SCD Type 1), handling deletes and out-of-order events.

### A) Regular Spark Streaming + Delta MERGE

```python
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number

target = "training.bronze.customer_current"   # must be created beforehand with the right schema
chk = "/Volumes/training/bronze/checkpoints/customer_current"

def upsert_batch(batch_df, batch_id):
    # 1. A micro-batch can contain several changes for one key -> keep the latest
    w = Window.partitionBy("c_custkey").orderBy(col("event_ts").desc())
    latest = (batch_df.withColumn("rn", row_number().over(w))
                      .filter("rn = 1").drop("rn"))

    # 2. MERGE: deletes, updates (only if newer), inserts
    (DeltaTable.forName(spark, target).alias("t")
        .merge(latest.alias("s"), "t.c_custkey = s.c_custkey")
        .whenMatchedDelete(condition="s.operation = 'DELETE'")
        .whenMatchedUpdateAll(condition="s.operation != 'DELETE' AND s.event_ts > t.event_ts")
        .whenNotMatchedInsertAll(condition="s.operation != 'DELETE'")
        .execute())

(spark.readStream.table("training.bronze.customer_changes")
    .writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", chk)
    .trigger(availableNow=True)
    .start()
    .awaitTermination())
```

### B) Declarative pipeline with Auto CDC

```python
from pyspark import pipelines as dp
from pyspark.sql.functions import expr

dp.create_streaming_table("customer_current")     # table name: customer_current (first argument), schema inferred

dp.create_auto_cdc_flow(
    target="customer_current",                       # must match the name above
    source="customer_changes",                       # name of the source table/view
    keys=["c_custkey"],
    sequence_by="event_ts",                          # ordering column
    apply_as_deletes=expr("operation = 'DELETE'"),
    except_column_list=["operation"],                # don't store the CDC op column
    stored_as_scd_type=1,                            # change to 2 for full history
)
```

### What changed

| Concern | Streaming + MERGE | Auto CDC |
|---|---|---|
| Duplicate keys in one micro-batch | Window + `row_number()` by hand | Handled |
| Out-of-order events | `event_ts > t.event_ts` condition by hand | Handled via `sequence_by` |
| Late event for an already-deleted key | Can wrongly re-insert the row | Handled (deletes are tracked) |
| Target table | Create in advance, manage schema; full name in `target = "training.bronze.customer_current"` | `dp.create_streaming_table("customer_current")` — name is the first argument |
| Checkpoint | You manage it | Managed |
| SCD Type 2 (history) | Much more MERGE logic (close old row, insert new) | `stored_as_scd_type=2` — adds `__START_AT` / `__END_AT` |
| Lines of logic | ~25 | ~10 |

## 6. Medallion Architecture in SDP

SDP is designed around the medallion (bronze/silver/gold) pattern:

```
  Source                Bronze                  Silver                  Gold
 (CSV files)       (raw ingestion)         (cleaned/enriched)     (aggregated)

 abfss://...  ──►  customer_bronze  ──►  customer_silver  ──►  customer_gold
  (Auto Loader)    Streaming Table      Streaming Table        Materialized View
                   • raw schema         • typed columns         • aggregations
                   • all rows           • nulls dropped         • business metrics
                   • _rescued_data      • enriched fields       • dashboard ready
```

### Why each layer matters

| Layer | Purpose | SDP type |
|---|---|---|
| **Bronze** | Exact copy of source data; never transform, never drop rows | Streaming table (Auto Loader) |
| **Silver** | Clean, type-correct, enriched; expectations enforced | Streaming table |
| **Gold** | Business aggregations and KPIs; queryable by BI tools | Materialized view |

### Pipeline dependency graph
SDP automatically resolves the execution order from your `spark.read.table()` / `spark.readStream.table()` references:
```
Auto Loader scan ─► customer_bronze ─► customer_silver ─► customer_gold
```
You never write `depends_on` manually — SDP infers it.

### Where the names come from in this example

| Layer | Table name | Defined by |
|---|---|---|
| Bronze | `customer_bronze` | `@dp.table(name="customer_bronze")` |
| Silver | `customer_silver` | `@dp.table(name="customer_silver")`, reads `spark.readStream.table("customer_bronze")` |
| Gold | `customer_gold` | `@dp.materialized_view(name="customer_gold")`, reads `spark.read.table("customer_silver")` |

**Why a materialized view and not a streaming table?** An aggregate changes when new orders arrive for an existing month, so the result must be recomputed for consistency. A materialized view (batch `spark.read.table`) does that automatically; a streaming table only appends new rows.


## 7. Data Quality with Expectations

Expectations are **declarative data quality rules** attached to a table. SDP evaluates them on every row and records pass/fail metrics in the event log.

### Three enforcement modes

| Decorator | Row fails expectation → | Use when |
|---|---|---|
| `@dp.expect(name, expr)` | Row is kept, failure is **logged** (warn) | You want to track quality without blocking |
| `@dp.expect_or_drop(name, expr)` | Row is **dropped** from the table | Invalid rows must not pollute downstream |
| `@dp.expect_or_fail(name, expr)` | Entire **pipeline fails** | Zero-tolerance; data must be perfect |

### Examples

```python
@dp.expect("no rescued data", "_rescued_data IS NULL")          # warn only
@dp.expect_or_drop("valid_key",  "c_custkey IS NOT NULL")        # drop bad rows
@dp.expect_or_drop("positive_balance", "c_acctbal >= -999")      # business rule
@dp.expect_or_fail("valid_segment",  "c_mktsegment IS NOT NULL") # must never be null
@dp.table(comment="Silver: cleaned customer data")              # no name= -> table name is customer_silver
def customer_silver():
    return spark.readStream.table("customer_bronze")
```

> 📛 The first argument of every `expect*` decorator (e.g. `"valid_key"`) is the **expectation name** shown in the Data quality tab — it is not a table name.

### Where to see expectation metrics
In the Pipeline UI → click a table node → **Data quality** tab shows pass/fail counts per expectation per run.

## 8. This Training's Pipeline

This training uses a **configuration-driven** approach: all tuneable parameters live in a JSON config file, and a single Python file defines bronze and silver tables for **two datasets** (customer + orders), plus a **gold materialized view** for orders.

The pipeline is written with the `dp` API (`from pyspark import pipelines as dp`).

### `config/sdp_pipeline_config_dp.json` — configuration
All source paths, schemas, table names, expectations, and silver transforms are defined here — **not hardcoded in Python**.
* **Customer** tables: `customer_bronze` → `customer_silver` (CSV from `abfss://inboundfiles@unityms.dfs.core.windows.net/customer`)
* **Orders** tables: `orders_bronze` → `orders_silver` → `orders_gold_monthly` (CSV from `abfss://inboundfiles@unityms.dfs.core.windows.net/orders`)

> In the config the names carry a `dp_` prefix (e.g. `dp_orders_gold_monthly`) so they don't clash with other pipelines in the same schema.

### Gold config block (`orders.gold`)

| Key | Value | Purpose |
|---|---|---|
| `table_name` | `dp_orders_gold_monthly` | Name of the materialized view |
| `date_column` | `o_orderdate` | Date used to build the period |
| `period` | `month` | Aggregation grain (`date_trunc` unit) |
| `period_column_name` | `order_month` | Name of the period output column |
| `key_columns` | `["o_custkey"]` | Group-by key(s) |
| `aggregations` | `order_count`, `total_price`, `avg_price`, `min_price`, `max_price` | Output column → SQL aggregate expression on `o_totalprice` / `o_orderkey` |
| `expectations` | warn `non_negative_total`, drop `valid_month` | Quality rules on the aggregated rows |

To aggregate by week or by another key, change `period` or `key_columns` in the config; to add a metric, add an entry to `aggregations`. No code change needed.

### `sdp_customer_pipeline_dp.py` — pipeline code
* Uses `from pyspark import pipelines as dp` and `@dp.table()`, `@dp.materialized_view()`, `dp.expect*()`
* Reads pipeline tables with `spark.readStream.table("table_name")` (silver) and `spark.read.table("table_name")` (gold, batch)
* Builds expectation decorators from the config in `_apply_expectations()`
* **Table names come from the config**, not from the function names: `@dp.table(name=cust_bronze["table_name"], ...)`

### Where the table names are defined

| Table | Config key (`sdp_pipeline_config_dp.json`) | Used in code |
|---|---|---|
| `customer_bronze` | `bronze.table_name` | `@dp.table(name=cust_bronze["table_name"])` |
| `customer_silver` | `silver.table_name` | `@dp.table(name=cust_silver["table_name"])`, reads `spark.readStream.table(cust_bronze["table_name"])` |
| `orders_bronze` | `orders.bronze.table_name` | `@dp.table(name=ord_bronze["table_name"])` |
| `orders_silver` | `orders.silver.table_name` | `@dp.table(name=ord_silver["table_name"])`, reads `spark.readStream.table(ord_bronze["table_name"])` |
| `orders_gold_monthly` | `orders.gold.table_name` | `@dp.materialized_view(name=ord_gold["table_name"])`, reads `spark.read.table(ord_silver["table_name"])` |

To rename a table, change `table_name` in the config — no code change needed.
* Aligns with the open-source Apache Spark Declarative Pipelines standard

### How they connect

```
 sdp_pipeline_config_dp.json  ◄── read by ──►  sdp_customer_pipeline_dp.py
        │
        ▼
 ┌── customer ─────────────┐    ┌── orders ───────────────┐
 │ customer_bronze          │    │ orders_bronze            │
 │       │                  │    │       │                  │
 │       ▼                  │    │       ▼                  │
 │ customer_silver          │    │ orders_silver            │
 │                          │    │       │                  │
 │                          │    │       ▼                  │
 │                          │    │ orders_gold_monthly (MV) │
 └─────────────────────────┘    └──────────────────────────┘
```



## 9. Decision Matrix — When to Choose SDP vs Regular Streaming

### Quick decision matrix

| Scenario | Choose | Why |
|---|---|---|
| **100s of tables replicated across layers** (bronze → silver → gold) with little or the **same transformation** | **SDP** | One config-driven loop defines every table; independent tables run **in parallel** in one update; adding a table = adding a config entry |
| Upserts / deletes from a CDC feed, SCD Type 1 or 2 | **SDP** (Auto CDC) | Out-of-order events, duplicates and history handled without MERGE code |
| Data quality rules with visible metrics | **SDP** | Expectations record pass / warn / drop counts per run |
| Team wants one pipeline to deploy, monitor and full-refresh | **SDP** | One DAG, one UI, one event log |
| Write to custom / external systems per micro-batch (REST API, JDBC, multiple targets with custom logic) | **Regular streaming** | `foreachBatch` gives full control of the write |
| Very low latency, per-stream trigger or real-time mode tuning | **Regular streaming** | You control trigger, cluster and query settings per stream |
| Complex custom stateful logic or arbitrary side effects | **Regular streaming** | Full Structured Streaming API without pipeline constraints |
| Existing, stable streaming job that already works | **Keep regular streaming** | Migrate only when the benefits above are needed |
| One-off or ad-hoc small load | **Regular Spark job** | A pipeline adds no value for a single run |

### Scenario: 100s of tables with common transformation

**Regular streaming** — one `writeStream` per table, per layer:
* you create and track a checkpoint path for every table and layer (e.g. 300 tables × 2 layers = 600 checkpoints)
* you start the queries and manage ordering (bronze before silver) yourself, often as many job tasks
* adding a table = new code / new job task + new checkpoint

**SDP** — one loop over a config list:
```python
for cfg in CFG["sources"]:      # 1 entry per table in JSON
    create_bronze(cfg)          # @dp.table ... Auto Loader
    create_silver(cfg)          # @dp.table ... common transforms + expectations
    if "gold" in cfg:
        create_gold(cfg)        # @dp.materialized_view
```
* SDP builds **one DAG**; tables that do not depend on each other **run in parallel** automatically
* each table's bronze → silver order is inferred from `spark.readStream.table()`
* **adding a new table = add one JSON entry**, no code change (this is how this training's pipeline works, see Section 8)
* the same expectations and transforms apply to every table consistently
* one place to monitor, retry and full-refresh